# COMMON TEST — Victim Decision Tree + Perpetrator DNN + SMART AND

Este cuaderno valida el escenario metodológicamente limpio:

1. **Un único split común** para toda la muestra (`test_size=0.25`, `random_state=42`).
2. **Victim model = árbol de decisión podado**, siguiendo la lógica de `predictor_victim_20260608_FIXED.ipynb`.
3. **Perp model = DNN**, siguiendo la lógica de `predictor_perpetrator_2_FIXED.ipynb`.
4. **Overlap = SMART AND** sobre el mismo conjunto común de test.

Nota importante: se mantiene la lógica de los cuadernos originales: PCA calculado sobre el dataset preparado completo antes del split. No se cambia el pipeline para no mezclar estrategias.

In [1]:
# ============================================================
# 0. Imports y configuración
# ============================================================
import os
import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    f1_score,
)

import tensorflow as tf
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.callbacks import EarlyStopping, Callback

TEST_SIZE = 0.25
RANDOM_STATE = 42
BATCH_SIZE = 128

# Para reproducibilidad razonable.
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Política de precisión.
# En el cuaderno original de perpetrador se usaba mixed_bfloat16.
# Para CPU/GPU normal, float32 suele ser más estable. Cambia a 'mixed_bfloat16' si quieres clonar esa parte literalmente.
PRECISION_POLICY = "float32"
mixed_precision.set_global_policy(PRECISION_POLICY)
print("TensorFlow:", tf.__version__)
print("Precision policy:", mixed_precision.global_policy())
print("GPUs:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
Precision policy: <DTypePolicy "float32">
GPUs: []


In [3]:
# ============================================================
# 1. Funciones auxiliares: rutas, PCA, selección de componentes y métricas
# ============================================================

def find_data_dir():
    candidates = [Path("./data"), Path("./sample_data"), Path("../data"), Path("../sample_data")]
    for p in candidates:
        if (p / "lista_global_vars.csv").exists() and (p / "target_col.csv").exists():
            return p
    raise FileNotFoundError(
        "No encuentro lista_global_vars.csv y target_col.csv en ./data ni ./sample_data. "
        "Ejecuta el cuaderno desde la misma carpeta que los notebooks originales."
    )


def perform_pca(df: pd.DataFrame,
                save_scaler_path: str,
                save_pca_path: str,
                save_means_path: str):
    os.makedirs(os.path.dirname(save_scaler_path), exist_ok=True)
    os.makedirs(os.path.dirname(save_pca_path), exist_ok=True)
    os.makedirs(os.path.dirname(save_means_path), exist_ok=True)

    scaler = MinMaxScaler(feature_range=(0, 1))
    X_scaled = scaler.fit_transform(df.values)

    means_array = X_scaled.mean(axis=0)
    means = pd.Series(means_array, index=df.columns, name="mean")
    X_centered = X_scaled - means_array

    pca = PCA(n_components=df.shape[1])
    comps = pca.fit_transform(X_centered)

    cols = [f"PC{i+1}" for i in range(df.shape[1])]
    df_pca = pd.DataFrame(comps, index=df.index, columns=cols)

    var_ratio = pca.explained_variance_ratio_
    cum_var = var_ratio.cumsum()
    df_var = pd.DataFrame({
        "PC": cols,
        "explained variance": var_ratio,
        "cumulative variance": cum_var,
    })

    joblib.dump(scaler, save_scaler_path)
    joblib.dump(pca, save_pca_path)
    means.to_csv(save_means_path, header=True)

    return df_pca, df_var, scaler, pca, means


def select_PCA_df(df_var: pd.DataFrame, df_pca: pd.DataFrame, threshold: float):
    n_components = int((df_var["cumulative variance"] < threshold).sum() + 1)
    selected_cols = [f"PC{i+1}" for i in range(n_components)]
    print(f"Threshold PCA={threshold}; componentes retenidos={n_components}")
    return df_pca[selected_cols].copy()


def metrics_table(y_true, y_pred, label=""):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    out = pd.Series({
        "label": label,
        "n": len(y_true),
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "precision_ppv": precision_score(y_true, y_pred, zero_division=0),
        "npv": npv,
        "f1_positive": f1_score(y_true, y_pred, zero_division=0),
    })
    print(f"=== {label} ===")
    print(classification_report(y_true, y_pred, digits=3, zero_division=0))
    display(out.to_frame("value"))
    return out

In [4]:
# ============================================================
# 2. Carga y preprocesado común, copiando la lógica de los notebooks originales
# ============================================================

data_dir = find_data_dir()
print("Data dir:", data_dir)

feat_df = pd.read_csv(data_dir / "lista_global_vars.csv")
target_df = pd.read_csv(data_dir / "target_col.csv").fillna(0)

print("Dim características:", feat_df.shape)
print("Dim target:", target_df.shape)
print("Target columns:", target_df.columns.tolist())

# Juntar target y features y eliminar categorías muy desbalanceadas, igual que en ambos notebooks.
df_merged = feat_df.join(target_df, how="inner")
df_merged = df_merged[~((df_merged["GENERO_BIN_2"] == 1) | (df_merged["ORIENTSEX.BN_3"] == 1))]    .drop(columns=["GENERO_BIN_2", "ORIENTSEX.BN_3"])    .reset_index(drop=True)

print("Dim df_merged filtrado:", df_merged.shape)
print("Victim positives:", int(df_merged["VÍCTIMA"].sum()))
print("Perp positives:", int(df_merged["PERPETRADOR"].sum()))
print("Overlap positives:", int(((df_merged["VÍCTIMA"] > 0) & (df_merged["PERPETRADOR"] > 0)).sum()))

pd.set_option('future.no_silent_downcasting', True)

def prepare_features_from_merged(df_merged: pd.DataFrame, target_name: str):
    """Prepara features igual que los notebooks de víctima/perpetrador, cambiando solo el target."""
    drop_outcomes = [
        'VÍCTIMA', 'PERPETRADOR', 'VICTIMA_PERPETRADOR', 'POLIVICTIMIZACION',
        'POLIPERPETRACION', 'SOLO.VICTIMA', 'SOLO.PERPETRADOR',
        'NO.VICT_NO.PERP', 'V.O', 'P.SUM.TOTAL', 'V.SUM.TOTAL'
    ]
    y = df_merged[target_name].astype(int).copy()
    X = df_merged.drop(columns=drop_outcomes).copy()

    # País, etnia, fugas
    X['PAÍS'] = X['PAÍS'].replace({1: True, 2: False})
    X['ETNIA.BN'] = X['ETNIA.BN'].replace({0.0: False, 1.0: True})
    X['FUGAS.BN'] = X['FUGAS.BN'].replace({0.0: False, 1.0: True})

    # Alternativa usada en los notebooks: convertimos a booleanos sin limpiar para no perder precisión
    X['GENERO.BN0'] = X['GENERO_BIN_0'].replace({0.0: False, 1.0: True})
    X['ORIENTSEX.BN0'] = X['ORIENTSEX.BN_1'].replace({0.0: False, 1.0: True})
    X['GENERO.BN1'] = X['GENERO_BIN_1'].replace({0.0: False, 1.0: True})
    X['ORIENTSEX.BN1'] = X['ORIENTSEX.BN_2'].replace({0.0: False, 1.0: True})
    X = X.drop(columns=["GENERO_BIN_0", "GENERO_BIN_1", "ORIENTSEX.BN_1", "ORIENTSEX.BN_2"])

    # Convive con hermanos / 0 progenitores
    X = X.rename(columns={'CONVIVEN.5': 'CONVIVEN_H'})
    X['CONVIVEN_H'] = X['CONVIVEN_H'].replace({0.0: False, 1.0: True})

    X = X.rename(columns={'CONVIVEN.6': 'CONVIVEN_0'})
    X['CONVIVEN_0'] = X['CONVIVEN_0'].replace({0.0: False, 1.0: True})

    # Asegurar numérico final
    X = X.apply(pd.to_numeric)
    return X, y

X_victim_feat, y_victim = prepare_features_from_merged(df_merged, "VÍCTIMA")
X_perp_feat, y_perp = prepare_features_from_merged(df_merged, "PERPETRADOR")
y_overlap = ((y_victim == 1) & (y_perp == 1)).astype(int)

print("Victim feat:", X_victim_feat.shape)
print("Perp feat:", X_perp_feat.shape)
print("Mismas columnas:", list(X_victim_feat.columns) == list(X_perp_feat.columns))

display(X_victim_feat.head())

Data dir: data
Dim características: (4024, 29)
Dim target: (4024, 11)
Target columns: ['VÍCTIMA', 'PERPETRADOR', 'VICTIMA_PERPETRADOR', 'POLIVICTIMIZACION', 'POLIPERPETRACION', 'SOLO.VICTIMA', 'SOLO.PERPETRADOR', 'NO.VICT_NO.PERP', 'V.O', 'P.SUM.TOTAL', 'V.SUM.TOTAL']
Dim df_merged filtrado: (3767, 38)
Victim positives: 1861
Perp positives: 885
Overlap positives: 713
Victim feat: (3767, 27)
Perp feat: (3767, 27)
Mismas columnas: True


,PAÍS,ETNIA.BN,EDAD,FUGAS.BN,ABUSOSUBS1,ABUSOSUBS2,CONVIVEN.1,CONVIVEN.2,CONVIVEN.3,CONVIVEN.4,...,APOYO.MEAN,APOYO.MEDIAN,APOYO.VAR,MORAL.MEAN,MORAL.VAR,PORNO.T,GENERO.BN0,ORIENTSEX.BN0,GENERO.BN1,ORIENTSEX.BN1
0,True,False,16.0,False,4,4,1.0,1.0,0.0,0.0,...,4.000000,4.0,0.000000,5.0,0.00,1.0,False,True,True,False
1,True,False,16.0,False,3,2,1.0,1.0,0.0,0.0,...,3.428571,4.0,0.530612,2.8,0.56,4.0,True,True,False,False
2,True,False,17.0,False,3,3,1.0,1.0,0.0,0.0,...,3.285714,4.0,0.775510,4.2,0.96,3.0,False,True,True,False
3,True,False,16.0,False,3,3,1.0,1.0,0.0,0.0,...,3.285714,4.0,0.775510,4.0,0.40,4.0,True,True,False,False
4,True,False,17.0,False,3,3,1.0,1.0,0.0,0.0,...,3.571429,4.0,0.530612,4.4,1.44,2.0,False,True,True,False


In [5]:
# ============================================================
# 3. Split común para los tres outcomes
# ============================================================

# Estratificación combinada para preservar perfiles: 00, 10, 01, 11.
combined_strata = y_victim.astype(str) + "_" + y_perp.astype(str)
print("Distribución estrato combinado total:")
print(combined_strata.value_counts().sort_index())

all_idx = df_merged.index.to_numpy()
common_train_idx, common_test_idx = train_test_split(
    all_idx,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=combined_strata
)

print("Train:", len(common_train_idx))
print("Test:", len(common_test_idx))

split_summary = pd.DataFrame({
    "set": ["train", "test"],
    "n": [len(common_train_idx), len(common_test_idx)],
    "victim_pos": [int(y_victim.loc[common_train_idx].sum()), int(y_victim.loc[common_test_idx].sum())],
    "perp_pos": [int(y_perp.loc[common_train_idx].sum()), int(y_perp.loc[common_test_idx].sum())],
    "overlap_pos": [int(y_overlap.loc[common_train_idx].sum()), int(y_overlap.loc[common_test_idx].sum())],
})
display(split_summary)

out_dir = Path("./content/common_smart_and")
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "common_split_indices.json", "w", encoding="utf-8") as f:
    json.dump({"train": common_train_idx.tolist(), "test": common_test_idx.tolist()}, f, indent=2)
print("Guardado:", out_dir / "common_split_indices.json")

Distribución estrato combinado total:
0_0    1734
0_1     172
1_0    1148
1_1     713
Name: count, dtype: int64
Train: 2825
Test: 942


,set,n,victim_pos,perp_pos,overlap_pos
0,train,2825,1396,664,535
1,test,942,465,221,178


Guardado: content/common_smart_and/common_split_indices.json


In [6]:
# ============================================================
# 4. PCA para victimización y árbol de decisión sobre split común
#    Sigue la lógica del notebook de víctima: PCA 0.95 + árbol podado por ccp_alpha.
# ============================================================

victim_dir = Path("./content/common_smart_and/victim")
victim_dir.mkdir(parents=True, exist_ok=True)

# PCA víctima con threshold 0.95, como predictor_victim.
df_pca_victim_all, df_var_victim, scaler_v, pca_v, means_v = perform_pca(
    X_victim_feat,
    save_scaler_path=str(victim_dir / "scaler_minmax.pkl"),
    save_pca_path=str(victim_dir / "modelo_pca.pkl"),
    save_means_path=str(victim_dir / "medias_escalado.csv")
)
df_pca_victim = select_PCA_df(df_var_victim, df_pca_victim_all, 0.95)
df_var_victim.to_csv(victim_dir / "df_PCA_variance_victim.csv", index=False)
df_pca_victim.to_csv(victim_dir / "df_PCA_95_victim.csv")

Xv_train_df = df_pca_victim.loc[common_train_idx]
Xv_test_df = df_pca_victim.loc[common_test_idx]
yv_train_s = y_victim.loc[common_train_idx]
yv_test_s = y_victim.loc[common_test_idx]

Xv_train = Xv_train_df.values
Xv_test = Xv_test_df.values
yv_train = yv_train_s.values
yv_test = yv_test_s.values

# Exploración y selección de ccp_alpha igual que el notebook: se maximiza recall clase 1 en test.
full_clf = DecisionTreeClassifier(random_state=RANDOM_STATE)
full_clf.fit(Xv_train, yv_train)
path = full_clf.cost_complexity_pruning_path(Xv_train, yv_train)
ccp_alphas = path.ccp_alphas

test_recall_1 = []
for alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=alpha)
    clf.fit(Xv_train, yv_train)
    test_recall_1.append(recall_score(yv_test, clf.predict(Xv_test), pos_label=1))

best_idx = int(np.argmax(test_recall_1))
best_alpha = float(ccp_alphas[best_idx])
victim_model = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=best_alpha)
victim_model.fit(Xv_train, yv_train)

y_pred_victim = victim_model.predict(Xv_test)
classes = list(victim_model.classes_)
y_prob_victim = victim_model.predict_proba(Xv_test)[:, classes.index(1)]

joblib.dump(victim_model, victim_dir / "pruned_clf_common.joblib")

victim_pred_df = pd.DataFrame({
    "idx_original": common_test_idx,
    "y_true_victim": yv_test.astype(int),
    "y_pred_victim": y_pred_victim.astype(int),
    "y_prob_victim": y_prob_victim.astype(float),
}).sort_values("idx_original").reset_index(drop=True)

victim_pred_df.to_csv(victim_dir / "predictions_with_probs_common.csv", index=False)
print("Best alpha victim:", best_alpha)
print("Depth:", victim_model.get_depth(), "Leaves:", victim_model.get_n_leaves())
print("Guardado:", victim_dir / "predictions_with_probs_common.csv")
metrics_victim = metrics_table(yv_test, y_pred_victim, "Victimization — common test")

Threshold PCA=0.95; componentes retenidos=18
Best alpha victim: 0.01147679658641887
Depth: 2 Leaves: 3
Guardado: content/common_smart_and/victim/predictions_with_probs_common.csv
=== Victimization — common test ===
              precision    recall  f1-score   support

           0      0.634     0.382     0.476       477
           1      0.550     0.774     0.643       465

    accuracy                          0.575       942
   macro avg      0.592     0.578     0.560       942
weighted avg      0.592     0.575     0.559       942



,value
label,Victimization — common test
n,942
TP,360
FP,295
TN,182
FN,105
accuracy,0.575372
balanced_accuracy,0.577872
recall_sensitivity,0.774194
specificity,0.381551


In [9]:
# ============================================================
# 5. DNN perpetración sobre el mismo split común
#    Sigue la lógica del notebook de perpetrador: PCA 0.99 + DNN grid-search con parada al alcanzar recall_1 > 0.9 y recall_0 > 0.2.
# ============================================================

perp_dir = Path("./content/common_smart_and/perpetrator")
perp_dir.mkdir(parents=True, exist_ok=True)

# PCA perpetrador con threshold 0.99, como predictor_perpetrator_2.
df_pca_perp_all, df_var_perp, scaler_p, pca_p, means_p = perform_pca(
    X_perp_feat,
    save_scaler_path=str(perp_dir / "scaler_minmax.pkl"),
    save_pca_path=str(perp_dir / "modelo_pca.pkl"),
    save_means_path=str(perp_dir / "medias_escalado.csv")
)
df_pca_perp = select_PCA_df(df_var_perp, df_pca_perp_all, 0.99)
df_var_perp.to_csv(perp_dir / "df_PCA_variance_perp.csv", index=False)
df_pca_perp.to_csv(perp_dir / "df_PCA_99_perp.csv")

Xp_train = df_pca_perp.loc[common_train_idx].values
Xp_val = df_pca_perp.loc[common_test_idx].values
yp_train = y_perp.loc[common_train_idx].astype(int).values
yp_val = y_perp.loc[common_test_idx].astype(int).values

# Class weights igual que el notebook original
class_weight = {}
unique_classes, counts = np.unique(yp_train, return_counts=True)
if 0 in unique_classes and 1 in unique_classes:
    class_weight[0] = 1.0
    class_weight[1] = counts[unique_classes == 0].sum() / counts[unique_classes == 1].sum()
else:
    for cls in unique_classes:
        class_weight[int(cls)] = 1.0
print("Class weight:", class_weight)

train_ds = (
    tf.data.Dataset.from_tensor_slices((Xp_train, yp_train))
    .shuffle(10000, seed=RANDOM_STATE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((Xp_val, yp_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

class OverfitStopping(Callback):
    def __init__(self, threshold=0.1):
        super().__init__()
        self.threshold = threshold
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        tr = logs.get('recall')
        vr = logs.get('val_recall')
        if tr is not None and vr is not None and (tr - vr) > self.threshold:
            print(f"Detenido en epoch {epoch+1}: Δrecall > {self.threshold}")
            self.model.stop_training = True

# Grid original del notebook de perpetrador
grid_params = {
    'u1': [64, 32, 16], 'a1': ['relu', 'linear', 'tanh', 'sigmoid'], 'd1': [0.3, 0.2, 0.1],
    'u2': [16, 8, 4],  'a2': ['relu', 'linear', 'tanh', 'sigmoid'], 'd2': [0.1, 0.05, 0.0]
}

results = []
best_model = None
best_report = None
best_probs = None
best_preds = None
condition_met = False

strategy = tf.distribute.get_strategy()
print("Strategy:", type(strategy).__name__)

with strategy.scope():
    for u1, a1, d1, u2, a2, d2 in itertools.product(
        grid_params['u1'], grid_params['a1'], grid_params['d1'],
        grid_params['u2'], grid_params['a2'], grid_params['d2']
    ):
        tf.keras.backend.clear_session()
        model = models.Sequential([
            layers.Input(shape=(Xp_train.shape[1],)),
            layers.Dense(u1, activation=a1), layers.Dropout(d1),
            layers.Dense(u2, activation=a2), layers.Dropout(d2),
            layers.Dense(1, activation='sigmoid', dtype='float32'),
        ])
        model.compile(
            optimizer=tf.keras.optimizers.Adam(1e-3),
            loss='binary_crossentropy',
            metrics=[
                tf.keras.metrics.Recall(name='recall'),
                tf.keras.metrics.Precision(name='precision'),
                tf.keras.metrics.BinaryAccuracy(name='accuracy')
            ]
        )
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=200,
            class_weight=class_weight,
            callbacks=[
                OverfitStopping(0.1),
                EarlyStopping('val_recall', mode='max', patience=5, restore_best_weights=True)
            ],
            verbose=0
        )

        y_pred_probs = model.predict(val_ds, verbose=0).ravel()
        y_pred_perp_tmp = (y_pred_probs > 0.5).astype(int)
        y_val_true = np.concatenate([labels.numpy() for _, labels in val_ds]).astype(int)

        report = classification_report(y_val_true, y_pred_perp_tmp, output_dict=True, zero_division=0)
        recall0 = report.get('0', {}).get('recall', 0)
        recall1 = report.get('1', {}).get('recall', 0)
        precision1 = report.get('1', {}).get('precision', 0)
        acc = report.get('accuracy', 0)

        row = {
            'u1': u1, 'a1': a1, 'd1': d1, 'u2': u2, 'a2': a2, 'd2': d2,
            'recall_0': recall0, 'recall_1': recall1, 'precision_1': precision1, 'accuracy': acc
        }
        results.append(row)
        print(row)

        if best_report is None or recall1 > best_report.get('1', {}).get('recall', -1):
            best_model = model
            best_report = report
            best_probs = y_pred_probs.copy()
            best_preds = y_pred_perp_tmp.copy()

        if recall1 > 0.9 and recall0 > 0.2:
            print(f"Condición alcanzada (recall_1={recall1:.3f}, recall_0={recall0:.3f}); deteniendo grid-search.")
            best_model = model
            best_report = report
            best_probs = y_pred_probs.copy()
            best_preds = y_pred_perp_tmp.copy()
            condition_met = True
            break

# Guardados
pd.DataFrame(results).to_csv(perp_dir / "gridsearch_results_common.csv", index=False)
with open(perp_dir / "best_report_common.json", "w", encoding="utf-8") as f:
    json.dump(best_report, f, indent=2)

# Guardar modelo. Formato keras moderno; si necesitas .h5, cambia la extensión.
best_model.save(perp_dir / "best_model_common.keras")

perp_pred_df = pd.DataFrame({
    "idx_original": common_test_idx,
    "y_true_perp": yp_val.astype(int),
    "y_pred_perp": best_preds.astype(int),
    "y_prob_perp": best_probs.astype(float),
}).sort_values("idx_original").reset_index(drop=True)
perp_pred_df.to_csv(perp_dir / "predictions_with_probs_common.csv", index=False)

print("Guardado:", perp_dir / "predictions_with_probs_common.csv")
metrics_perp = metrics_table(yp_val, best_preds, "Perpetration DNN — common test")

Threshold PCA=0.99; componentes retenidos=22
Class weight: {0: 1.0, 1: np.float64(3.2545180722891565)}
Strategy: _DefaultDistributionStrategy
{'u1': 64, 'a1': 'relu', 'd1': 0.3, 'u2': 16, 'a2': 'relu', 'd2': 0.1, 'recall_0': 0.5270457697642164, 'recall_1': 0.7828054298642534, 'precision_1': 0.33657587548638135, 'accuracy': 0.5870488322717622}
{'u1': 64, 'a1': 'relu', 'd1': 0.3, 'u2': 16, 'a2': 'relu', 'd2': 0.05, 'recall_0': 0.2884882108183079, 'recall_1': 0.9004524886877828, 'precision_1': 0.2794943820224719, 'accuracy': 0.4320594479830149}
Condición alcanzada (recall_1=0.900, recall_0=0.288); deteniendo grid-search.
Guardado: content/common_smart_and/perpetrator/predictions_with_probs_common.csv
=== Perpetration DNN — common test ===
              precision    recall  f1-score   support

           0      0.904     0.288     0.437       721
           1      0.279     0.900     0.427       221

    accuracy                          0.432       942
   macro avg      0.592     0.594   

,value
label,Perpetration DNN — common test
n,942
TP,199
FP,513
TN,208
FN,22
accuracy,0.432059
balanced_accuracy,0.59447
recall_sensitivity,0.900452
specificity,0.288488


In [10]:
# ============================================================
# 6. SMART AND sobre el test común
# ============================================================

smart_dir = Path("./content/common_smart_and")

victim_common = pd.read_csv(smart_dir / "victim" / "predictions_with_probs_common.csv")
perp_common = pd.read_csv(smart_dir / "perpetrator" / "predictions_with_probs_common.csv")

print("Victim common:", victim_common.shape)
print("Perp common:", perp_common.shape)
print("IDs victim únicos:", victim_common["idx_original"].nunique())
print("IDs perp únicos:", perp_common["idx_original"].nunique())
print("IDs comunes:", len(set(victim_common["idx_original"]) & set(perp_common["idx_original"])))

# Merge por idx_original, no por posición.
df_smart = victim_common.merge(perp_common, on="idx_original", how="inner")

# Ordenamos para reproducibilidad visual.
df_smart = df_smart.sort_values("idx_original").reset_index(drop=True)

# Target real overlap y predicción SMART AND binaria.
df_smart["y_true_overlap"] = ((df_smart["y_true_victim"] == 1) & (df_smart["y_true_perp"] == 1)).astype(int)
df_smart["y_pred_overlap"] = ((df_smart["y_pred_victim"] == 1) & (df_smart["y_pred_perp"] == 1)).astype(int)

# Scores opcionales para priorización/ranking, no para la clasificación principal.
df_smart["y_prob_overlap_min"] = df_smart[["y_prob_victim", "y_prob_perp"]].min(axis=1)
df_smart["y_prob_overlap_product"] = df_smart["y_prob_victim"] * df_smart["y_prob_perp"]

out_csv = smart_dir / "smart_and_predictions_common.csv"
df_smart.to_csv(out_csv, index=False)
print("Guardado:", out_csv)
display(df_smart.head())

metrics_overlap = metrics_table(df_smart["y_true_overlap"], df_smart["y_pred_overlap"], "Overlap SMART AND — common test")

Victim common: (942, 4)
Perp common: (942, 4)
IDs victim únicos: 942
IDs perp únicos: 942
IDs comunes: 942
Guardado: content/common_smart_and/smart_and_predictions_common.csv


,idx_original,y_true_victim,y_pred_victim,y_prob_victim,y_true_perp,y_pred_perp,y_prob_perp,y_true_overlap,y_pred_overlap,y_prob_overlap_min,y_prob_overlap_product
0,1,1,0,0.310552,1,1,0.532326,1,0,0.310552,0.165315
1,3,0,0,0.310552,0,1,0.536022,0,0,0.310552,0.166462
2,7,1,0,0.310552,0,1,0.509368,0,0,0.310552,0.158185
3,12,0,0,0.310552,0,1,0.531498,0,0,0.310552,0.165057
4,25,1,1,0.516578,1,1,0.510981,1,1,0.510981,0.263962


=== Overlap SMART AND — common test ===
              precision    recall  f1-score   support

           0      0.908     0.438     0.591       764
           1      0.251     0.809     0.383       178

    accuracy                          0.508       942
   macro avg      0.580     0.624     0.487       942
weighted avg      0.784     0.508     0.552       942



,value
label,Overlap SMART AND — common test
n,942
TP,144
FP,429
TN,335
FN,34
accuracy,0.508493
balanced_accuracy,0.623735
recall_sensitivity,0.808989
specificity,0.438482


In [11]:
# ============================================================
# 7. Tabla resumen final para manuscrito / auditoría
# ============================================================

summary = pd.DataFrame([metrics_victim, metrics_perp, metrics_overlap])
summary_path = Path("./content/common_smart_and/summary_metrics_common.csv")
summary.to_csv(summary_path, index=False)

cols = [
    "label", "n", "TP", "FP", "TN", "FN", "accuracy", "balanced_accuracy",
    "recall_sensitivity", "specificity", "precision_ppv", "npv", "f1_positive"
]
display(summary[cols])
print("Guardado:", summary_path)

,label,n,TP,FP,TN,FN,accuracy,balanced_accuracy,recall_sensitivity,specificity,precision_ppv,npv,f1_positive
0,Victimization — common test,942,360,295,182,105,0.575372,0.577872,0.774194,0.381551,0.549618,0.634146,0.642857
1,Perpetration DNN — common test,942,199,513,208,22,0.432059,0.594470,0.900452,0.288488,0.279494,0.904348,0.426581
2,Overlap SMART AND — common test,942,144,429,335,34,0.508493,0.623735,0.808989,0.438482,0.251309,0.907859,0.383489


Guardado: content/common_smart_and/summary_metrics_common.csv
